
Geo Data Science with Python,
Prof. Susanna Werth, VT Geosciences

# Time Series Analysis


This notebook is accompanied by the lecture L08 presentation slides.


---


Content:
-------
- **A.** Basics: Smoothing, Auto- & Cross-Correlations
- **B.** Fourier Transform & Harmonic Models
- **C.** Using Real Data

--- 

In [ ]:

# On Google Colab, install the following packages...

# ! pip install cftime # needed in part C
# ! pip install xarray #(only if original data are downloaded again)


In [ ]:

# Import standard packages
import requests
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy
from scipy.ndimage import gaussian_filter1d

# let's set a standard size for figures (a bit smaller, so they fit on the screen)
from matplotlib import rcParams
rcParams['figure.figsize'] = [4, 3]
rcParams.update({'font.size': 12})


# Generate Synthetic Time Series Data
... for demonstrations

In [ ]:

# Create a signal: combination of two sine waves + noise
fs = 500  # sampling frequency (Hz=1/s)
T = 2     # duration (seconds)
t = np.linspace(0, T, int(fs*T), endpoint=False)

# signal =  20Hz + 120Hz + random noise
f1, f2 = 50, 120     # frequencies of sine waves
amp1, amp2, amp3 = 1, 0.5, 0.5  # amplitudes of sine waves and noise
signal = amp2*np.sin(2*np.pi*f1*t) + amp1*np.sin(2*np.pi*f2*t)
signal += signal + amp3 * np.random.randn(len(t))

plt.figure(figsize=(10,3))
plt.plot(t, signal)
plt.title("Original Signal (time domain)")
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.show()


In Geosciences, we would work with slightly longer time periods and interested in lower frequencies, so let's our example alter a bit by redefining the units of the signal and the amplitudes. 

Let's generate a time series, that is multiple years long, contains some annual variation, some longer term variation and some noise:

In [ ]:

# Create a signal: combination of annual and semi-annual sine waves + noise
fs2 = 365  # sampling frequency (1/year = 31.5 Mega Hz)
T2 = 6     # duration (years)
t2 = np.linspace(0, T2, int(fs2*T2))

# signal = 1/year + 1/(3years) + random noise
f1, f2 = 1, 1/3     # frequencies of sine waves
amp1, amp2, amp3 = 1, 0.6, 0.5  # amplitudes of sine waves, let's say this is [m]
signal2 = amp1*np.sin(2*np.pi*f1*t2) + amp2*np.sin(2*np.pi*f2*t2)
signal2_noise = signal2 + amp3  * np.random.randn(len(t2))

plt.figure(figsize=(6,2))
plt.plot(t2, signal2_noise,label='noise added')
plt.plot(t2, signal2,label='synthetic signal')
plt.title("Original Signal (time domain)")
plt.xlabel("Time [year]")
plt.ylabel("Amplitude")
plt.legend()
plt.show()


**Task**: What are the sampling rate and the units of our time vector `t2`?


---
# A. Basics


### Smoothing/Filtering

Smoothing or filtering a time series reduces high-frequency noise to better reveal underlying trends and patterns in the data. It improves interpretability by averaging or attenuating rapid fluctuations while preserving the broader structure of the signal.


In [ ]:

# Simple moving average smoother
window_size = 2 # n-day window
smoothed_ma = np.convolve(signal2_noise, np.ones(window_size)/window_size, mode='same')
# Note: Units of window are same as the dataset, here days, since we work with dayly sampled data

# Gaussian smoother
smoothed_gauss = gaussian_filter1d(signal2_noise, sigma=2)
# Note: Gaussian sigma stands for half filter width, where weights reach value of 0.5
#       For example, a Gaussian sigma of 7 is comparable to a window filter of 30

# Plot the time series stacked on top of each other
fig, ax = plt.subplots(figsize=(10,3))
ax.plot(t2, signal2        + 0 , 'k', label='synthetic')
ax.plot(t2, signal2_noise  + 2 , 'blue', label='noisy')
ax.plot(t2, smoothed_ma    + 4 , 'r', label='window')
ax.plot(t2, smoothed_gauss + 6 , 'orange', label='gaussian')
ax.set_yticks([0, 2, 4, 6])
ax.set_yticklabels(['0', '+2', '+4', '+6'])
plt.title("Smoothing effect (time domain)")
plt.xlabel("Time [Year]")
plt.ylabel("Amplitude [m]")
plt.legend()
plt.show()


### Autocorrelation

Autocorrelation measures how strongly a time series is related to a shifted (lagged) version of itself, revealing repeating patterns or persistence over time. It is useful for detecting periodicity, trend dependence, and whether current values are influenced by past behavior.


In [ ]:

from statsmodels.tsa.stattools import acf

ac, confint = acf(signal2_noise,nlags=365*3, alpha=0.05)  # returns autocorrelation values
lags = np.arange(len(ac))

plt.figure()
plt.fill_between(lags, confint[:, 0], confint[:, 1],
                color='lightblue', alpha=0.4, label='95% CI')
plt.stem(lags[::15], ac[::15], label='auto-R') # using stemplot, plot every 15th value
plt.title("Autocorrelation")
plt.xlabel("Lag [days]")
plt.ylabel("Correlation")
plt.legend(loc='lower right')
plt.show()


Note: the light blue background shows uncertainty of the correlation estimate at 95% confidence interval.

### Cross-Correlation

Cross-correlation measures how strongly two different time series are related at different time lags, indicating whether changes in one signal lead or follow changes in the other. It is commonly used to identify causal timing relationships, synchronization, or shared patterns between paired variables.


In [ ]:
# Create a new signal with a phase shift

phase_shift = -0.5 # 1/year

# signal = 1/year + 180/year + random noise
f1, f2 = 1, 1/3     # frequencies of sine waves
amp1, amp2, amp3 = 1, 0.6, 0.5  # amplitudes of sine waves, let's say this is [m]
signal3 = amp1*np.sin(2*np.pi*f1*t2+phase_shift*np.pi) + amp2*np.sin(2*np.pi*f2*t2)
signal3_noise = signal3 + amp3  * np.random.randn(len(t2))

plt.figure(figsize=(6,2))
plt.plot(t2, signal3_noise,'g',label='noise added')
plt.plot(t2, signal3,'orange',label='new signal')
plt.plot(t2, signal2,'blue',label='first signal')
plt.title("Phase-shifted Signal (time domain)")
plt.xlabel("Time [year]")
plt.ylabel("Amplitude")
plt.legend()
plt.show()


Note that, after removing pi/2 inside the sinus function, the phase-shifted synthetic signal is slightly delayed compared to the first synthetic signal.

Although statsmodels provides a cross-correlation function (`ccf`), we are now using pandas, because it allows us to include negative lags.


In [ ]:
# Estimating cross correlation with Pandas

series2 = pd.Series(signal2_noise)
series3 = pd.Series(signal3_noise)
nlags=365*2
lags = np.arange(-nlags, nlags+1)

cc = [series2.corr(series3.shift(l)) for l in lags]
# here series 3 is shifted with respect to series 2, which is repeated in a for loop

cc = np.squeeze(cc)
cc.shape

# Plot cross-corrlation results
plt.figure(figsize=(10,3))
plt.stem(lags[::15], cc[::15], label='auto-R') # using stemplot, plot every 15th value
plt.title("Cross-Correlation")
plt.xlabel("Lag [days]")
plt.ylabel("Correlation")
plt.legend(loc='lower right')
plt.show()


In [ ]:

# Find location with maximum correlation
idx = np.unravel_index(np.argmax(cc), cc.shape)
idx2 = np.unravel_index(np.argmin(cc), cc.shape)
print("Max correlation ", round(np.max(cc),2), " at lag of ", lags[idx], "days.")
print("Min correlation ", round(np.min(cc),2), " at lag of ", lags[idx2], "days.")


**Task.** From the cross-correlation example between signal 2 and 3 above, we find a maximum correlation of 0.71 at a lag of -83 days. Given the negative sign of the lag, which signal (2 or 3) peaks earlier, and why?

---

## Exercise A

1. When does smoothing reveal important structure — and when can it hide or distort key information? Alter the smoothing factor (use either gaussian or window filter), plot some examples. Then discuss what are reasonable values for the synthetic signal above. How would you choose an appropriate smoothing scale for the phenomena you study?

2. Estimate the autocorrelation for a time series containing only noise and another time series consisting only of a sine (or cosine) function. Explain your findings.

3. Discuss how you could distinguish whether a peak in cross-correlation indicates a true physical lead-lag relationship, or is simply due to common oscillations (due to a common driver, e.g., annual variations) in the signals? Would any form of signal pre-processing be useful?

**Extra Credit**

4. Discuss the differences and similarities between auto- and cross-correlation. Would it make sense to use negative lags in both cases?

---

---
# B. Frequency Analysis and Harmonic Models



### Fast Frourier Transform (FFT)

Scipy and numpy come with a `np/scipy.fft.fft()` function. 
This function computes the one-dimensional n-point discrete Fourier Transform (DFT) with an efficient Fast Fourier Transform (FFT) algorithm.

In [ ]:

# Compute Fourier Transform using scipy or numpy fft package:
fhat = scipy.fft.fft (signal2_noise)     # estimates fft for regularly sampled time series (no time vector needed)
fft_freq = scipy.fft.fftfreq(len(t2), 1/fs2) # gets discrete (!) FFT sample frequency and scales units (here 1/year)


The raw FFT output contains complex numbers that include phase. Therefor the FFT output contains negative values. However, for real-valued time series, the FFT amplitude spectrum is symmetric, with the negative frequency side mirroring the positive frequency side. Therefore, we save and plot only the positive half of the spectrum, which contains all unique frequency information.

In [ ]:

# Keeps only the positive frequencies (real signal symmetry)
positive = fft_freq > 0     # filter for positive frequencies
freqs = fft_freq[positive]

# reconstruct amplitudes & phases from fft output
N = len(t2)
amplitude = np.abs(fhat[positive]) * 2 / N 
# phase = np.angle(fft_values) # reconstruct phase form fft output (not further required here)


In [ ]:

# Plot the signal spectrum
fig,ax = plt.subplots(1,2,figsize=(10,3))
ax[0].stem(freqs, amplitude, basefmt=" ")
ax[0].set_title("Amplitude Spectrum over f")
ax[0].set_xlabel("Frequency [31MHz = 1/year]")
ax[0].set_ylabel("Amplitude")
ax[1].stem(1/freqs, amplitude, basefmt=" ")
ax[1].set_title("Amplitude Spectrum over T=1/f")
ax[1].set_xlabel("Period [year]")
plt.show()


The frequency spectrum is shown to the left, it is equally sampled. On the right, frequency is converted to signal period T = 1/f, which I find a bit more intuitive to understand. Instead of number of cycle per time unit, it indicates the full length of the cycle in a time unit. However, here you can also see, that while f is uniformly sampled, this is not the case for T, because T = 1/f.

### Inverse FFT

Once we have the spectrum, we can choose or filter frequencies that we like or find important and reconstruct only those components of the signal. This allows us, for example, to isolate specific signal components, such as an annual signal, or remove noise from the dataset.

In [ ]:

# Reconstruction: Inverse Fourier Transform
f_reconstructed = scipy.fft.ifft(fhat)

# Complex part in the reconstruction is due to rounding errors in the numeric
# solution, constist of very small residuals that can be removed
f_reconstructed = np.real(f_reconstructed) 


In [ ]:

# Plot original signal and reconstruction
plt.figure(figsize=(6,3))
plt.plot(t2, signal2_noise, label='Original Signal', linewidth=1)
plt.plot(t2, f_reconstructed+2, label='Fully Reconstructed', linewidth=1)
plt.xlabel("Time [year]")
plt.ylabel("Amplitude")
plt.title("Signal Reconstruction from Dominant Fourier Components")
plt.legend()
plt.show()


### Filtering via Inverse FFT

In [ ]:

# Redo FFT to make sure we have the correct data in the variables
fhat = scipy.fft.fft (signal2_noise)     # estimates fft for regularly sampled time series (no time vector needed)
freq = scipy.fft.fftfreq(len(t2), 1/fs2) # gets discrete (!) FFT sample frequency and scales units (here 1/year)
positive = np.where(freq > 0)
amplitude = np.abs(fhat[positive])

# Find the k largest peaks
k=2
topK = np.argsort(amplitude)[-k:]        # indices of top 2
topK_freqs = freq[positive][topK]  # their frequencies
print("Top frequencies (Hz):", topK_freqs)

# Keep only those peaks and their conjugates in fhat
# Note: the negative side has to be filtered equally
fhat_filtered = np.zeros_like(fhat)
for ki in topK:
    idx = positive[0][ki]
    fhat_filtered[idx] = fhat[idx]          # positive side
    fhat_filtered[-idx] = fhat[-idx]        # matching negative side

# Inverse FFT to reconstruct the low-pass filtered signal
f_reconstructed_filt = np.fft.ifft(fhat_filtered).real


In [ ]:

# Plot original signal and reconstruction
plt.figure(figsize=(6,3))
plt.plot(t2, signal2_noise, 'black',label='Original Signal', linewidth=2)
plt.plot(t2, f_reconstructed, label='Reconstructed', linewidth=0.5)
plt.plot(t2, f_reconstructed_filt, label=f'Filtered (top {k} freqs)', linewidth=1)
plt.xlabel("Time [year]")
plt.ylabel("Amplitude")
plt.title("Signal Reconstruction from Dominant Fourier Components")
plt.legend()
plt.show()


**Task**: Manually change the number of reconstructed components to see how the signal reconstruction changes.


### Harmonic Models: Signal Decomposition

If you do not have regularly sampled data or gaps in the data, or you want to identify parameters of very specific oscillations, a Fourier transform might not be the ideal approach. In that case, we can construct a linear model that contains the amplitudes of harmonic waves. This allows us to use linear regression for fitting oscillating signal models to the data, e.g., annual signals.


In [ ]:
from sklearn.linear_model import LinearRegression

y = signal2_noise

# Create Data Matrix X
P1 = 1    # 1 year
w1 = 2*np.pi / P1
X = np.column_stack([ t2, np.sin(w1*t2), np.cos(w1*t2)]) # use this X matrix, if you want to fit only an annual wave

# Create model object and fit model
model = LinearRegression()  # fits y = bias + a0*t + a1*sin(2pi/T*t) + a2*sin(2pi/T*t) 
model.fit(X, y)

# Predict y from model
y_est = model.predict(X)

# Predict only 1 year signal component from model
bias = model.intercept_
a0 = model.coef_[0]
a1 = model.coef_[1]
a2 = model.coef_[2]
y_est_1year = bias + a1*np.sin(w1*t2) + a2*np.cos(w1*t2)       # 1-year component
y_est_trend = bias + a0*t2    # linear component

# Print results
print("Ampl sine/cosine (coef):", model.coef_[0], model.coef_[1])       # a1
print("Intercept (bias):", model.intercept_) # a0 automatically added to the model
print("R² score:", model.score(X, y))        # R2 Coefficient of determination


In [ ]:
# Plot original signal and modeled
plt.figure(figsize=(10,3))
plt.plot(t2, signal2_noise, label='Original Signal', linewidth=1)
plt.plot(t2, signal2_noise-y_est, label='Remainder', linewidth=0.5)
plt.plot(t2, y_est, label='Full Model', linewidth=2)
plt.plot(t2, y_est_1year, label='Annual Period', linewidth=2)
plt.plot(t2, y_est_trend, label='Linear Trend', linewidth=2)
plt.xlabel("Time [years]")
plt.ylabel("Amplitude")
plt.title("Harmonic Regression Model and Components")
plt.legend(loc='lower left')
plt.show()

---

## Exercise B

1. Increase the noise of the synthetic time series, show examples. Discuss: At which noise level (approximately) does it become difficult to resolve the correct amplitude of the frequencies for the annual (1/year) and 3-year periods (1/3year frequencies)?

2. Design a step function, e.g., with the code below, and add some noise. What happens to the frequency analysis with FFT ? Discuss if FFT is well designed to identify or even handle steps in the datasets. If you are unsure, discuss with your peers and/or use an LLM application to investigate an answer to this question, then summarize your findings in a brief answer, using your own words.
```python
    signal_step = np.ones(len(t2)) * amp1
    signal_step[~(t2>T2/2)] -= amp1 * 2
```

**Extra Credit**

3. Include an additional 3-year period in the harmonic regression model and solve it. Then, add a plot of the 3-year-only period in the result plot, similar as it is done above for the 1-year period. Discuss the differences you get for the linear trend when including or not including the 3-year period.


---
# C. Using Real Data

## GNSS Data: Land Motion

In [ ]:

# Load GNSS Data
url = ('https://raw.githubusercontent.com/GeoPythonVT/geosf25_material/main/data/gnssData.csv')
gnss_df = pd.read_csv(url, index_col=0)
gnss_df.head(5)


In [ ]:

# Plot data
plt.figure(figsize=(6,4))
plt.plot(gnss_df["decyear"], gnss_df["east(m)"]  - gnss_df["east(m)"][0] , label="east(m)")
plt.plot(gnss_df["decyear"], gnss_df["north(m)"] - gnss_df["north(m)"][0], label="north(m)")
plt.plot(gnss_df["decyear"], gnss_df["up(m)"]    - gnss_df["up(m)"][0]   , label="up(m)")
plt.xlabel("Year")
plt.ylabel("Up (m)")
plt.title("GNSS Land Surface Deformation in California")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:

# Need to interpolate the time series to a regular time vector without gaps.

y  = gnss_df['up(m)'].to_numpy()  # values

# Create uniformly sampled time vector in datenum, decimal years and datetime
dec = gnss_df['decyear'].to_numpy()    # your decimal years
dtnm = np.round((dec-2020)*365.25)     # days since 2020,1,0
dtnm_dly = np.arange(0, dtnm[-1])
dec_dly = dtnm_dly/365.25+2020
t0 = pd.Timestamp('2020-1-1')
dt_index_dly = t0 + pd.to_timedelta(dtnm_dly, unit='D')
dt_index_dly = pd.to_datetime(dt_index_dly)  # ensure DateTimeIndex

# Build linear interpolator
from scipy.interpolate import interp1d
f = interp1d(dtnm, y, kind='linear', bounds_error=False, fill_value='extrapolate')
y_interp = f(dtnm_dly)


In [ ]:
# Store data in new Pandas DataFrame
gnss_dly = pd.DataFrame({
    "datenum": dtnm_dly,
    "decyear": dec_dly,
    "up(m)": y_interp
})

gnss_dly.index = pd.to_datetime(dt_index_dly); gnss_dly = gnss_dly.sort_index()
gnss_dly.head(5)


In [ ]:

# Estimate monthly mean values
gnss_mnthly = gnss_dly.resample("MS").mean() 
gnss_mnthly.index = gnss_mnthly.index + pd.offsets.Day(14) # move timestamp to mid month
gnss_mnthly.head(5)


In [ ]:

#Plot interpolated data
plt.figure(figsize=(6,4))
plt.plot(dec, y-y[0] , label="Irregular",linewidth=2)
plt.plot(dec_dly, y_interp  - y_interp[0]    , label="Regular",linewidth=1)
plt.plot(gnss_mnthly["decyear"], gnss_mnthly["up(m)"]  - gnss_mnthly["up(m)"].iloc[0],label="Monthly",linewidth=1)
plt.xlabel("Year")
plt.ylabel("Up (m)")
plt.title("GNSS Land Surface Deformation in California")
plt.legend()
plt.grid(True)
plt.show()


## GRACE Data: Terrestrial Water Storage Change

In [ ]:
import requests

# Download GRACE Data
url = ('https://raw.githubusercontent.com/GeoPythonVT/geosf25_material/main/data/graceData.npz')
out = "graceData.npz"
r = requests.get(url)
open(out, "wb").write(r.content)


In [ ]:

# Load GRACE time series
data = np.load("graceData.npz", allow_pickle=True)  # allow_pickle if objects/strings
lwe_ts = data["array1"] # lwe stands for liquid water equivalent (height of a water column)
time = data["array2"]
txt = data["labels"]

# Store data in new Pandas DataFrame
grace_df = pd.DataFrame({
    "lwe": lwe_ts
})
grace_df.index = pd.to_datetime(time); grace_df = grace_df.sort_index()

# Get regular monthly time vector
time_mnthly = pd.date_range(time[0], time[-1], freq='MS') + pd.offsets.Day(14)
time_mnthly 

# Interpolate unregularily sampled monthly data to regularly monthly time vector
grace_mnthly = (
    grace_df.reindex(grace_df.index.union(time_mnthly))          # keep originals + target mids
     .interpolate(method='time')           # time-based linear interpolation
     .reindex(time_mnthly)                         # pick only mids
)
grace_mnthly.head(5)


In [ ]:

# Plot GRACE data
plt.figure(figsize=(6,4))
plt.plot(grace_df.index, grace_df['lwe'], label='Irregular',linewidth=3 )
plt.plot(grace_mnthly.index, grace_mnthly['lwe'], label='Regular',linewidth=1   )
plt.xlabel("Time")
plt.ylabel("TWS variation (cm)")
plt.title("Terrestrial Water Storage Change in California")
plt.grid(True)
plt.show()


---

Note, the final dataset are stored in the following variables:

- gnss_dly: GNSS data resampled to regular daily time interval (2020/1/1-2025/7/27)
- gnss_mnthly: GNSS data averaged to regular monthly time interval (2020/1/15-2025/7/15) 
- grace_mnthly: GRACE data resampled to regular monthly time interval (2002/5/15-2025/8/15)



---
## Exercise C

1. According to the sampling theorem, what is the highest resolvable frequency from the GNSS daily and the GRACE monthly datasets? And what do you expect would be the longest period you can resolve from the datasets? Explain your answers.  

2. Use the following code relying on Pandas to remove the trend in both time series. Conduct autocorrelation analysis with all three signals (daily GNSS, monthly GNSS, and monthly GRACE), before and after detrending. What is the major difference between the results for a) with/without trend, b) daily/monthly GNSS, and c) monthly GNSS/GRACE? Discuss how the trend and the sampling rate influence the autocorrelation results.  

```python
import scipy.signal as signal
detrended = pd.Series(signal.detrend(gnss_dly['up(m)']), index=gnss_dly.index)
```

3. Conduct FFT analysis with **detrended** GRACE and GNSS (either daily or monthly) signals. Then discuss the following:
    - What are the dominant frequencies/periods in the signal? 
    - How reliable do you consider the results for the lowest resulting frequency from the FFT?
    - Pick one dataset and respective FFT output, reconstruct a certain number of the largest frequency components (note that you need to integrate the dataset, time series vector, and the unit of the time series consistently in the code to get the right units for the frequencies and reconstructed time series). Then estimate the residual between the original and the reconstructed time series. How many components do you need to reconstruct so that the residuals are free of annual variations?


4. Unlike the synthetic data we worked with earlier, the real-world data are not stationary (even after detrending). How does this reflect in the results of the FFT? Discuss this.

**Extra Credit**

5. Conduct cross-correlation analysis. Note that the observation period is different. Use the GNSS monthly data and reduce the GRACE time series to the same time interval before conducting the analysis. Use detrended data. Discuss the results.

6. Conduct a harmonic regression, extracting an annual, a 3-year and a 4-year period from the GRACE time series. What do you observe?


---
# Supplement: Code for original data download

### GNSS Data

In [ ]:
# Download GNSS Data

# # GNSS site KDL1 data 
# url = "https://geodesy.unr.edu/gps_timeseries/IGS20/tenv3/NA/KDL1.NA.tenv3"

# with requests.get(url, stream=True, verify=True) as r:
#     r.raise_for_status()
#     with open("KDL1.NA.tenv3", "wb") as f:
#         f.write(r.content)

# gnss_df = pd.read_csv("KDL1.NA.tenv3", sep='\s+', header=0)
# # save only the coordinate columns
# gnss_df = gnss_df[["yyyy.yyyy", "__east(m)", "_north(m)", "____up(m)"]].rename(columns={"yyyy.yyyy": "decyear", "__east(m)": "east(m)", "_north(m)":"north(m)", "____up(m)": "up(m)"})
# gnss_df.head()

# # More info about the GNSS site: https://geodesy.unr.edu/NGLStationPages/stations/KDL1.sta

# Save data frame in csv file
# gnss_df.to_csv("gnssData.csv", index=True) # index=True keeps the row index


### GRACE Data

In [ ]:
# This is code used to download GRACE data
# raw_url='https://download.csr.utexas.edu/outgoing/grace/RL0603_mascons/CSR_GRACE_GRACE-FO_RL0603_Mascons_all-corrections.nc'
# out = "CSR_GRACE_GRACE-FO_RL0603_Mascons_all-corrections.nc"
# with requests.get(raw_url, stream=True, verify=False) as r:
#     r.raise_for_status()
#     with open(out, "wb") as f:
#         for chunk in r.iter_content(chunk_size=1024*1024): # use this download version for large files
#             if chunk:
#                 f.write(chunk)

# # More info about the data: https://www2.csr.utexas.edu/grace/RL06_mascons.html


In [ ]:
# This is the code used to extract GRACE data

# from netCDF4 import num2date   # on google colab you have to install this
# import xarray as xr
# import cftime


# # Load Data with xarray instead of NetCDF4
# grace_ds = xr.open_dataset("CSR_GRACE_GRACE-FO_RL0603_Mascons_all-corrections.nc")
# print([ e for e in grace_ds.variables ]) # lists all dimensions in the dataset

# # Load Data
# lon = grace_ds["lon"].values
# lon = ((lon + 180) % 360) - 180
# lat = grace_ds["lat"].values
# lwe = grace_ds["lwe_thickness"].values
# t = grace_ds["time"].values        # numeric time
# units = grace_ds["time"].Units     # Unit: "days since 2002-01-01T00:00:00Z"
# cal   = grace_ds["time"].attrs.get("calendar","standard")
# time = cftime.num2date(t, units, calendar=cal,
#                         only_use_cftime_datetimes=False,
#                         only_use_python_datetimes=True)

# # Target location (in Southern California)
# lat0, lon0 = 37, -120

# # Find nearest index
# ilat = np.abs(lat - lat0).argmin()
# ilon = np.abs(lon - lon0).argmin()

# print(f"Nearest grid cell index: lat={ilat}, lon={ilon}")
# print(f"Grid coordinates: ({lat[ilat]:.2f}, {lon[ilon]:.2f})")

# # Get time series at this location
# lwe_ts = lwe[:,ilat,ilon]

# # Save variables
# np.savez('graceData.npz', array1=lwe_ts, array2=time, labels=['lwe_ts','time'])
